In [4]:
import duckdb

In [5]:
duckdb.sql("SET TimeZone = 'UTC'")

SuccessBOOLEAN


chose 200MB for maximum_object_size because of the rule 2 x largest value you've seen (86MB)

In [41]:
print(duckdb.sql("DESCRIBE SELECT * FROM read_json('../data/bronze/prim_global/date=2026-11-14/hour=18/20261114T184926Z.json.gz', maximum_object_size=200000000)"))

┌─────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

to see the nesting and if each dict is just a dict or a list, used these cmdlines:

zcat data/bronze/prim_global/date=2026-11-14/hour=18/20261114T184926Z.json.gz > ~/snapshot.json
firefox ~/snapshot.json

firefox in json tab can show each nesting at a time

![Firefox JSON tree of a snapshot](images/firefox-json-tree.png)

In [42]:
f = "'../data/bronze/prim_global/date=2026-11-14/hour=18/20261114T184926Z.json.gz', maximum_object_size=200000000"

In [43]:
print(duckdb.sql(f"SELECT j.LineRef.value as line_ref, j.DatedVehicleJourneyRef.value as journey_ref FROM (SELECT unnest(Siri.ServiceDelivery.EstimatedTimetableDelivery[1].EstimatedJourneyVersionFrame[1].EstimatedVehicleJourney) AS j FROM read_json({f}) ) LIMIT 5"))
print(duckdb.sql(f"SELECT COUNT(*) as row_count FROM (SELECT unnest(Siri.ServiceDelivery.EstimatedTimetableDelivery[1].EstimatedJourneyVersionFrame[1].EstimatedVehicleJourney) FROM read_json({f}) )"))



┌────────────────────┬─────────────────────────────────────────────────────┐
│      line_ref      │                     journey_ref                     │
│      varchar       │                       varchar                       │
├────────────────────┼─────────────────────────────────────────────────────┤
│ STIF:Line::C01103: │ MeC_Bus_PC:VehicleJourney::20261114.53.A.C01103:LOC │
│ STIF:Line::C01107: │ MeC_Bus_PC:VehicleJourney::20261114.78.R.C01107:LOC │
│ STIF:Line::C01498: │ COESS:VehicleJourney::24918:LOC                     │
│ STIF:Line::C01229: │ MeC_Bus_PC:VehicleJourney::20261114.6.R.C01229:LOC  │
│ STIF:Line::C01096: │ MeC_Bus_PC:VehicleJourney::20261114.72.R.C01096:LOC │
└────────────────────┴─────────────────────────────────────────────────────┘

┌───────────┐
│ row_count │
│   int64   │
├───────────┤
│     12208 │
└───────────┘



In [44]:
print(duckdb.sql(f"SELECT j.LineRef.value as line_ref, j.DatedVehicleJourneyRef.value as journey_ref, k.StopPointRef.value as stop_ref, k.AimedDepartureTime as aimed_departure, k.ExpectedDepartureTime as expected_departure, k.DepartureStatus as departure_status FROM (select j, unnest(j.EstimatedCalls.EstimatedCall) AS k FROM (SELECT unnest(Siri.ServiceDelivery.EstimatedTimetableDelivery[1].EstimatedJourneyVersionFrame[1].EstimatedVehicleJourney) AS j FROM read_json({f}) )) LIMIT 15"))


┌────────────────────┬─────────────────────────────────────────────────────┬──────────────────────────┬──────────────────────────┬──────────────────────────┬──────────────────┐
│      line_ref      │                     journey_ref                     │         stop_ref         │     aimed_departure      │    expected_departure    │ departure_status │
│      varchar       │                       varchar                       │         varchar          │ timestamp with time zone │ timestamp with time zone │     varchar      │
├────────────────────┼─────────────────────────────────────────────────────┼──────────────────────────┼──────────────────────────┼──────────────────────────┼──────────────────┤
│ STIF:Line::C01103: │ MeC_Bus_PC:VehicleJourney::20261114.53.A.C01103:LOC │ STIF:StopPoint:Q:9114:   │ NULL                     │ 2026-11-14 19:04:35+00   │ ON_TIME          │
│ STIF:Line::C01103: │ MeC_Bus_PC:VehicleJourney::20261114.53.A.C01103:LOC │ STIF:StopPoint:Q:28023:  │ NULL       

In [45]:
print(duckdb.sql(f"SELECT COUNT(*) as count_all FROM (select j, unnest(j.EstimatedCalls.EstimatedCall) AS k FROM (SELECT unnest(Siri.ServiceDelivery.EstimatedTimetableDelivery[1].EstimatedJourneyVersionFrame[1].EstimatedVehicleJourney) AS j FROM read_json({f}) ))"))

print(duckdb.sql(f"SELECT COUNT(*) as aimed_departure_null FROM (select j, unnest(j.EstimatedCalls.EstimatedCall) AS k FROM (SELECT unnest(Siri.ServiceDelivery.EstimatedTimetableDelivery[1].EstimatedJourneyVersionFrame[1].EstimatedVehicleJourney) AS j FROM read_json({f}) )) WHERE k.AimedDepartureTime IS NULL"))


┌───────────┐
│ count_all │
│   int64   │
├───────────┤
│    101133 │
└───────────┘

┌──────────────────────┐
│ aimed_departure_null │
│        int64         │
├──────────────────────┤
│                31540 │
└──────────────────────┘



In [46]:
print(duckdb.sql(f"SELECT k.DepartureStatus as departure_status, COUNT(*) as number FROM (select j, unnest(j.EstimatedCalls.EstimatedCall) AS k FROM (SELECT unnest(Siri.ServiceDelivery.EstimatedTimetableDelivery[1].EstimatedJourneyVersionFrame[1].EstimatedVehicleJourney) AS j FROM read_json({f}) )) GROUP BY k.DepartureStatus ORDER BY number DESC"))


┌──────────────────┬────────┐
│ departure_status │ number │
│     varchar      │ int64  │
├──────────────────┼────────┤
│ ON_TIME          │  76122 │
│ NO_REPORT        │  10592 │
│ DELAYED          │   8613 │
│ NULL             │   3049 │
│ EARLY            │   1693 │
│ CANCELLED        │   1036 │
│ MISSED           │     21 │
│ ARRIVED          │      7 │
└──────────────────┴────────┘



In [59]:
print(duckdb.sql(f"SELECT j.LineRef.value as line_ref, j.DatedVehicleJourneyRef.value as journey_ref,  k.StopPointRef.value as stop_ref, k.AimedDepartureTime as aimed_departure, k.ExpectedDepartureTime as expected_departure, k.DepartureStatus as departure_status,  date_diff('second', expected_departure ,  aimed_departure) as delay_s FROM (select j, unnest(j.EstimatedCalls.EstimatedCall) AS k FROM (SELECT unnest(Siri.ServiceDelivery.EstimatedTimetableDelivery[1].EstimatedJourneyVersionFrame[1].EstimatedVehicleJourney) AS j FROM read_json({f}) )) WHERE delay_s IS NOT NULL ORDER BY delay_s DESC LIMIT 5"))


┌────────────────────┬─────────────────────────────────┬──────────────────────────┬──────────────────────────┬──────────────────────────┬──────────────────┬─────────┐
│      line_ref      │           journey_ref           │         stop_ref         │     aimed_departure      │    expected_departure    │ departure_status │ delay_s │
│      varchar       │             varchar             │         varchar          │ timestamp with time zone │ timestamp with time zone │     varchar      │  int64  │
├────────────────────┼─────────────────────────────────┼──────────────────────────┼──────────────────────────┼──────────────────────────┼──────────────────┼─────────┤
│ STIF:Line::C00090: │ COESS:VehicleJourney::24356:LOC │ STIF:StopPoint:Q:11735:  │ 2026-11-14 18:31:00+00   │ 2026-11-14 17:38:37+00   │ EARLY            │    3143 │
│ STIF:Line::C00573: │ COTMA:VehicleJourney::3426:LOC  │ STIF:StopPoint:Q:494389: │ 2026-11-14 19:43:00+00   │ 2026-11-14 18:53:01+00   │ EARLY            │    2999 

A way to write sql queries without having to repeat them each time:

In [17]:
calls = f"""
    WITH journeys AS (
        SELECT filename, unnest(Siri.ServiceDelivery.EstimatedTimetableDelivery[1].EstimatedJourneyVersionFrame[1].EstimatedVehicleJourney) AS j
        FROM read_json({f})
    ),
    calls AS (
        SELECT filename, j, unnest(j.EstimatedCalls.EstimatedCall) AS k 
        FROM journeys
    ),
    silver AS (
        SELECT j.LineRef.value as line_ref, j.DatedVehicleJourneyRef.value as journey_ref,  k.StopPointRef.value as stop_ref, 
        k.AimedDepartureTime as aimed_departure, k.ExpectedDepartureTime as expected_departure, k.DepartureStatus as departure_status,
        split_part(filename, '/', -1) as snapshot_file,
        CASE WHEN k.DepartureStatus IN ('NO_REPORT', 'CANCELLED') THEN NULL
            ELSE date_diff('second', k.AimedDepartureTime, k.ExpectedDepartureTime) END AS delay_s,
        CASE WHEN k.DepartureStatus = 'CANCELLED' THEN 'cancelled'
            WHEN k.DepartureStatus ='NO_REPORT' THEN 'no_report'
            WHEN k.AimedDepartureTime IS NULL THEN 'no_aimed'
            WHEN k.ExpectedDepartureTime IS NULL THEN 'no_expected'
            ELSE 'measured' END AS category
        FROM calls
    )
"""

In [7]:
#1
print(duckdb.sql(f"{calls} SELECT line_ref, stop_ref, aimed_departure, expected_departure, departure_status, delay_s FROM silver WHERE delay_s IS NOT NULL ORDER BY delay_s DESC LIMIT 5"))

┌────────────────────┬─────────────────────────┬──────────────────────────┬──────────────────────────┬──────────────────┬─────────┐
│      line_ref      │        stop_ref         │     aimed_departure      │    expected_departure    │ departure_status │ delay_s │
│      varchar       │         varchar         │ timestamp with time zone │ timestamp with time zone │     varchar      │  int64  │
├────────────────────┼─────────────────────────┼──────────────────────────┼──────────────────────────┼──────────────────┼─────────┤
│ STIF:Line::C00295: │ STIF:StopPoint:Q:27039: │ 2026-11-14 08:45:00+00   │ 2026-11-14 17:59:13+00   │ DELAYED          │   33253 │
│ STIF:Line::C00295: │ STIF:StopPoint:Q:27038: │ 2026-11-14 08:41:00+00   │ 2026-11-14 17:55:13+00   │ DELAYED          │   33253 │
│ STIF:Line::C00295: │ STIF:StopPoint:Q:27039: │ 2026-11-14 08:45:00+00   │ 2026-11-14 17:59:13+00   │ DELAYED          │   33253 │
│ STIF:Line::C00295: │ STIF:StopPoint:Q:27038: │ 2026-11-14 08:41:00+00   │ 

In [8]:
print(duckdb.sql(f"{calls} SELECT category, COUNT(*) as n_calls, round(100.0 * COUNT(*)/SUM(COUNT(*)) OVER(), 1) as percentage FROM silver GROUP BY category ORDER BY percentage DESC"))

┌─────────────┬─────────┬────────────┐
│  category   │ n_calls │ percentage │
│   varchar   │  int64  │   double   │
├─────────────┼─────────┼────────────┤
│ measured    │ 1864392 │       58.3 │
│ no_aimed    │  953714 │       29.8 │
│ no_report   │  334680 │       10.5 │
│ cancelled   │   33733 │        1.1 │
│ no_expected │   12263 │        0.4 │
└─────────────┴─────────┴────────────┘



In [9]:
print(duckdb.sql(f"""{calls} SELECT line_ref, COUNT(*) as n_calls, COUNT(delay_s) as n_measured,
       round(median(delay_s) / 60, 1) as median_delay_min,
       round(100.0 * COUNT(*) FILTER (WHERE departure_status = 'CANCELLED') / COUNT(*), 1) as cancel_rate
FROM silver GROUP BY line_ref HAVING COUNT(delay_s) >= 20
ORDER BY median_delay_min DESC LIMIT 10"""))

┌────────────────────┬─────────┬────────────┬──────────────────┬─────────────┐
│      line_ref      │ n_calls │ n_measured │ median_delay_min │ cancel_rate │
│      varchar       │  int64  │   int64    │      double      │   double    │
├────────────────────┼─────────┼────────────┼──────────────────┼─────────────┤
│ STIF:Line::C00579: │    3466 │       3330 │            143.8 │         0.0 │
│ STIF:Line::C00623: │      26 │         24 │             58.1 │         0.0 │
│ STIF:Line::C02770: │     300 │        300 │             46.2 │         0.0 │
│ STIF:Line::C01330: │    1856 │       1794 │             31.7 │         0.0 │
│ STIF:Line::C00480: │     564 │        453 │             24.7 │         0.0 │
│ STIF:Line::C02450: │     273 │        198 │             14.7 │         0.0 │
│ STIF:Line::C02449: │     564 │        467 │             14.1 │         0.0 │
│ STIF:Line::C02835: │    1900 │       1649 │             10.0 │         0.0 │
│ STIF:Line::C00473: │   11526 │       4024 │       

In [14]:
f = "'../data/bronze/prim_global/date=2026-11-14/hour=18/*.json.gz', maximum_object_size=200000000, filename=true"


In [18]:
print(duckdb.sql(f"""{calls} SELECT COUNT(*) as n_rows, COUNT(DISTINCT (journey_ref, stop_ref)) as n_calls,
       round(COUNT(*) / COUNT(DISTINCT (journey_ref, stop_ref)), 2) as snapshots_per_call FROM silver"""))


┌─────────┬─────────┬────────────────────┐
│ n_rows  │ n_calls │ snapshots_per_call │
│  int64  │  int64  │       double       │
├─────────┼─────────┼────────────────────┤
│ 3198782 │  204513 │              15.64 │
└─────────┴─────────┴────────────────────┘



In [19]:
# 2: one call seen in at least 10 snapshots, with a delay
print(duckdb.sql(f"""{calls}, seen_often AS (
    SELECT journey_ref, stop_ref FROM silver WHERE delay_s IS NOT NULL
    GROUP BY journey_ref, stop_ref HAVING COUNT(DISTINCT snapshot_file) >= 10 LIMIT 1)
SELECT snapshot_file, expected_departure, delay_s FROM silver JOIN seen_often USING (journey_ref, stop_ref) ORDER BY snapshot_file"""))

┌──────────────────────────┬──────────────────────────┬─────────┐
│      snapshot_file       │    expected_departure    │ delay_s │
│         varchar          │ timestamp with time zone │  int64  │
├──────────────────────────┼──────────────────────────┼─────────┤
│ 20261114T180001Z.json.gz │ 2026-11-14 19:24:00+00   │       0 │
│ 20261114T180201Z.json.gz │ 2026-11-14 19:24:00+00   │       0 │
│ 20261114T180401Z.json.gz │ 2026-11-14 19:24:00+00   │       0 │
│ 20261114T180601Z.json.gz │ 2026-11-14 19:24:00+00   │       0 │
│ 20261114T180801Z.json.gz │ 2026-11-14 19:24:00+00   │       0 │
│ 20261114T181002Z.json.gz │ 2026-11-14 19:24:00+00   │       0 │
│ 20261114T181201Z.json.gz │ 2026-11-14 19:24:00+00   │       0 │
│ 20261114T181401Z.json.gz │ 2026-11-14 19:24:00+00   │       0 │
│ 20261114T181601Z.json.gz │ 2026-11-14 19:24:00+00   │       0 │
│ 20261114T181802Z.json.gz │ 2026-11-14 19:24:00+00   │       0 │
│            ·             │           ·              │       · │
│         